In [1]:
# This cell is removed with the tag: "remove-input"
# As such, it will not be shown in documentation

import warnings
warnings.filterwarnings('ignore')

(Tutorial_Compare)=
# Compare

*Comparing molecular systems across selected attributes.*

The function {func}`molsysmt.basic.compare` allows you to compare two molecular systems, or specific parts of them, based on selected attributes. The function can return a single boolean or a detailed per-attribute report. Differences in array shapes or collection sizes are reported as `False` for the affected attributes; they are ordinary comparison results and do not emit warnings.

:::{versionadded} 1.0.0
:::

:::{admonition} API documentation
:class: dropdown

Follow this link for a detailed description of the input arguments, raised errors, and returned objects of this function: {func}`molsysmt.basic.compare`.
:::


Matching missing metadata entries compare equal, including `NaN` values used for undefined labels. A missing entry and a populated entry compare unequal. Use `output_type="dictionary"` to inspect which attributes differ.


## Basic usage

Let's explore how to compare molecular systems using three systems derived from the T4 lysozyme dataset (`181l.bcif.gz`):


In [2]:
import molsysmt as msm

In [3]:
molsys_A = msm.systems['T4 lysozyme L99A']['181l.bcif.gz']
molsys_B = msm.convert(molsys_A, to_form='molsysmt.MolSys')
molsys_C = msm.extract(molsys_B, selection='molecule_type=="protein"')

:::{note}
:class: dropdown

**Demo Systems Catalog**

This tutorial uses demonstration datasets provided by MolSysMT. To explore the full catalog of bundled systems, forms, and file paths, visit the {ref}`Demo Systems <user-foundations-entrance-demo-systems>` guide.
:::

Systems $A$ and $B$ are equivalent in content—they only differ in form (`mmcif` file vs `molsysmt.MolSys`). System $C$ is a subset containing only the protein part of $B$.

Let's inspect the state of $B$ and $C$ before comparing them:

In [4]:
msm.info(molsys_B)

form,n_atoms,n_groups,n_components,n_chains,n_molecules,n_entities,n_waters,n_ions,n_small_molecules,n_proteins,n_structures
molsysmt.MolSys,1441,302,141,6,141,5,136,2,2,1,1


In [5]:
msm.info(molsys_C)

form,n_atoms,n_groups,n_components,n_chains,n_molecules,n_entities,n_proteins,n_structures
molsysmt.MolSys,1289,162,1,1,1,1,1,1


Now let's compare $A$ with $B$, and $B$ with $C$:

In [6]:
msm.compare(molsys_A, molsys_B)

True

In [7]:
msm.compare(molsys_B, molsys_C)

False

:::{tip}
:class: dropdown

All methods defined in the {ref}`molsysmt.basic <API basic>` module can also be invoked from the library's top level. Hence, {func}`molsysmt.compare` is the same function as {func}`molsysmt.basic.compare`.
:::

By default, only the elements are compared. Other attributes such as `coordinates`, `box`, or `n_atoms` can be included explicitly in the comparison:

In [8]:
msm.compare(molsys_A, molsys_B, coordinates=True, box=True, n_atoms=True)

True

In [9]:
msm.compare(molsys_B, molsys_C, coordinates=True, box=True, n_atoms=True)

False

A detailed report showing the result for each attribute can be obtained by setting `output_type='dictionary'`:

In [10]:
msm.compare(molsys_B, molsys_C, coordinates=True, box=True, n_atoms=True, output_type='dictionary')

{'n_atoms': False, 'box': True, 'coordinates': False}

You can also test for the negation of a condition by setting the attribute value to `False`:

In [11]:
msm.compare(molsys_B, molsys_C, coordinates=False, box=True, n_atoms=False, output_type='dictionary')

{'n_atoms': True, 'box': True, 'coordinates': True}

## Comparing selections

You can restrict the comparison to specific atom selections in each system. For example, comparing the protein selection of $B$ against system $C$ (which is protein-only) yields `True`:

In [12]:
msm.compare(molsys_B, molsys_C, selection='molecule_type=="protein"', selection_2='all')

True

## Comparing structures

Similarly, when comparing multi-structure systems, use `structure_indices` for the first system and `structure_indices_2` for the second:

In [13]:
msm.compare(molsys_B, molsys_C, structure_indices=0, structure_indices_2=0)

False

## Comparing box quantities

The derived box attributes can be selected explicitly. Their quantities are
converted to the first operand's units before numerical comparison; dimensions
must be compatible. A quantity is not equated with an unlabelled numeric array.
Equal magnitudes in different physical dimensions are not equal observations.


In [14]:
import numpy as np
from molsysmt.native import Structures
from molsysmt import pyunitwizard as puw

molsys_A = Structures(box=puw.quantity(np.diag([1., 2., 3.])[None], 'nm'))
molsys_B = Structures(box=puw.quantity(np.diag([10., 20., 30.])[None], 'angstrom'))
assert msm.compare(molsys_A, molsys_B, box_lengths=True, box_angles=True, box_volume=True)
molsys_B = Structures(box=puw.quantity(np.diag([1., 2., 4.])[None], 'nm'))
assert not msm.compare(molsys_A, molsys_B, box_lengths=True)


:::{seealso}
:class: dropdown

**Related Tools & References**

- {ref}`Tutorial_Convert`: Convert a molecular system into another form or forms with {func}`molsysmt.basic.convert`.
- {ref}`Tutorial_Extract`: Extract a subset of atoms or structures from a molecular system into a new system with {func}`molsysmt.basic.extract`.
- {ref}`Tutorial_Info`: Print a summary of the contents, topology, and structural data of a molecular system with {func}`molsysmt.basic.info`.
:::